# Lightning Talk Deck — Exploratory Analysis

Native **Google Slides** -> per-slide structured tags, run piece by piece.

**Pipeline:** Slides API (exact text + server-side PNG thumbnails) -> vision model
tags each slide -> resumable JSONL -> forward-fill session dates -> table -> peeks.

**Backends:** flip `BACKEND` between a **local** model in LM Studio and **Claude**.
The exact text from the Slides API is fed into every call as grounding, so a small
local VLM leans on real text for names/titles instead of weak OCR.

**Setup**
- GCP project -> enable the *Google Slides API* -> create an *OAuth client (Desktop)* ->
  save as `credentials.json` next to this notebook.
- **LM Studio:** load a *vision* model (Qwen2.5-VL, Gemma 3, MiniCPM-V, ...) and start the
  local server (default `http://localhost:1234`).
- **Claude:** `export ANTHROPIC_API_KEY=...` before launching Jupyter.

### 0. Install deps (run once)

In [ ]:
%pip install -q anthropic openai google-api-python-client google-auth google-auth-oauthlib requests pandas networkx pyvis matplotlib

### 1. Config — pick your backend
Keep `CONCURRENCY = 1` for LM Studio (local inference serializes; parallel calls just
queue or risk OOM). Bump to ~6 only for the Claude backend.

In [ ]:
from pathlib import Path

BACKEND = "lmstudio"          # "lmstudio" or "anthropic"

# --- Google Slides ---
PRESENTATION = "https://docs.google.com/presentation/d/YOUR_PRESENTATION_ID/edit"  # paste your deck URL/ID
WORKDIR = Path("./lt_work"); WORKDIR.mkdir(parents=True, exist_ok=True)

# --- LM Studio (local) ---
LMSTUDIO_BASE_URL = "http://localhost:1234/v1"
MODEL_LOCAL = "internvl3-14b" # exact id from GET /v1/models. "" auto-picks data[0],
                              # whose order is NOT stable -> LM Studio JIT-loads the wrong model.
USE_STRUCTURED_OUTPUT = True  # force valid JSON via json_schema; turn off if the model chokes

# --- Anthropic (cloud) ---
MODEL_CLOUD = "claude-sonnet-4-6"   # verify current model string against the docs

# --- shared ---
THUMBNAIL_SIZE = "LARGE"      # LARGE ~1600px long edge; MEDIUM/SMALL also valid
TEMPERATURE = 0.0
MAX_TOKENS = 1024
CONCURRENCY = 1               # see note above

### 2. Connect to Google Slides (OAuth)
First run opens a browser for consent and caches `token.json`.

In [ ]:
import re
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

SCOPES = ["https://www.googleapis.com/auth/presentations.readonly"]
# If getThumbnail 403s, add "https://www.googleapis.com/auth/drive.readonly"
# and delete token.json to re-consent.

def get_service():
    token = WORKDIR / "token.json"
    creds = Credentials.from_authorized_user_file(str(token), SCOPES) if token.exists() else None
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file("credentials.json", SCOPES)
            creds = flow.run_local_server(port=0)
        token.write_text(creds.to_json())
    return build("slides", "v1", credentials=creds)

pres_id = re.search(r"/presentation/d/([\w-]+)", PRESENTATION)
pres_id = pres_id.group(1) if pres_id else PRESENTATION

service = get_service()
pres = service.presentations().get(presentationId=pres_id).execute()
slides = pres.get("slides", [])
print(f"{len(slides)} slides in: {pres.get('title', '?')}")

### 3. Extract exact text per slide
Recursively walks shapes, grouped elements, and tables.

In [ ]:
def collect_text(elements):
    out = []
    for el in elements or []:
        shape = el.get("shape")
        if shape and shape.get("text"):
            out += [te["textRun"]["content"]
                    for te in shape["text"].get("textElements", [])
                    if te.get("textRun")]
        if el.get("elementGroup"):
            out.append(collect_text(el["elementGroup"].get("children")))
        if el.get("table"):
            for row in el["table"].get("tableRows", []):
                for cell in row.get("tableCells", []):
                    if cell.get("text"):
                        out += [te["textRun"]["content"]
                                for te in cell["text"].get("textElements", [])
                                if te.get("textRun")]
    return "".join(out)

texts = {i: collect_text(s.get("pageElements")) for i, s in enumerate(slides, 1)}

# peek at a couple
for i in (1, len(slides)//2 or 1):
    print(f"--- slide {i} ---\n{texts[i][:300]!r}\n")

### 4. Fetch one thumbnail and preview it inline
Sanity-check legibility — especially that your **marker slides** are readable, since the dates are load-bearing.

In [ ]:
import requests
from IPython.display import Image, display

def fetch_thumb(idx):
    dst = (WORKDIR / "slides"); dst.mkdir(exist_ok=True)
    dst = dst / f"slide_{idx:04d}.png"
    if not dst.exists():
        t = service.presentations().pages().getThumbnail(
            presentationId=pres_id, pageObjectId=slides[idx-1]["objectId"],
            thumbnailProperties_mimeType="PNG",
            thumbnailProperties_thumbnailSize=THUMBNAIL_SIZE,
        ).execute()
        dst.write_bytes(requests.get(t["contentUrl"], timeout=60).content)
    return dst

TEST_SLIDE = 219
display(Image(filename=str(fetch_thumb(TEST_SLIDE))))

Fetch the rest (resumable — skips PNGs already on disk):

In [ ]:
png = {}
for i in range(1, len(slides) + 1):
    png[i] = fetch_thumb(i)
    if i % 25 == 0:
        print(f"thumbs {i}/{len(slides)}")
print("thumbnails ready")

### 5. The tagging call — test on ONE slide first
Get the prompt/schema right here before running the whole deck.

In [ ]:
import json

# Controlled topic vocabulary. ORDER IS MEANINGFUL: roughly least->most technical
# (personal/human -> domains -> data/engineering), so it reads as a gradient on the
# step-15 z-axis. "other" is the escape hatch (kept last; filtered out of the 3D viz).
# Specifics go in free-form `keywords`. Edit freely; reordering doesn't change validity.
TOPICS = [
    "personal & intro", "travel & places", "hobbies & crafts", "arts & culture",
    "health & medicine", "education & libraries", "career & workplace",
    "business & economics", "policy & government", "urban planning & transit",
    "environment & sustainability", "science & math", "tools & workflows",
    "data visualization", "data & databases", "ai & machine learning",
    "software & web dev", "other",
]
CONTENT_TYPES = ["text","screenshot","diagram","photo","chart","code","meme","video_still","mixed"]

PROMPT = f"""You are tagging one slide from a multi-year series of monthly "lightning talk" sessions. The deck has per-session MARKER slides (title cards announcing a month/session) and individual talk slides that USUALLY show the speaker's name somewhere.

You get the slide image AND the exact text from its text elements. Prefer the exact text for names/titles; use the image for what the text layer misses (screenshots, diagrams, photos).

Return ONLY a JSON object with these keys:
  is_marker      boolean. true ONLY for a session title/divider card.
  session_label  string|null. marker label as written, else null.
  session_date   string|null. ISO "YYYY-MM" if determinable, else null.
  speaker        string|null. presenter name if present. DO NOT GUESS -> null.
  title          string|null.
  summary        string. 1-2 sentences describing the TALK's content directly (present tense). Do NOT describe the slide itself — never write "this slide", "this presentation", "the slide shows/announces", "title card", etc. Just state what the talk is about.
  content_type   one of {",".join(CONTENT_TYPES)}.
  topics         array of 1-3 broad themes, EACH copied verbatim from this fixed list (no other values):
                 {", ".join(TOPICS)}.
                 Choose the best-fitting themes; use "other" only if none apply.
  keywords       array of up to 5 short, specific, lowercase tags for the actual subject
                 (e.g. "baroque guitar", "sonification", "rank choice voting"). [] if none.
  tech_mentioned array of named tools/technologies, or [].
  legible        boolean.
Output the JSON object and nothing else."""

SLIDE_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "is_marker": {"type": "boolean"},
        "session_label": {"type": ["string", "null"]},
        "session_date": {"type": ["string", "null"]},
        "speaker": {"type": ["string", "null"]},
        "title": {"type": ["string", "null"]},
        "summary": {"type": "string"},
        "content_type": {"type": "string", "enum": CONTENT_TYPES},
        "topics": {"type": "array", "items": {"type": "string", "enum": TOPICS}, "maxItems": 3},
        "keywords": {"type": "array", "items": {"type": "string"}, "maxItems": 5},
        "tech_mentioned": {"type": "array", "items": {"type": "string"}},
        "legible": {"type": "boolean"},
    },
    "required": ["is_marker","session_label","session_date","speaker","title",
                 "summary","content_type","topics","keywords","tech_mentioned","legible"],
}

def parse_json(raw):
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("```", 2)[1].lstrip("json").strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return json.loads(raw[raw.index("{"): raw.rindex("}") + 1])

Initialize the chosen backend's client:

In [ ]:
import base64

if BACKEND == "anthropic":
    from anthropic import Anthropic
    client = Anthropic()
    active_model = MODEL_CLOUD
else:
    from openai import OpenAI
    client = OpenAI(base_url=LMSTUDIO_BASE_URL, api_key="lm-studio")
    active_model = MODEL_LOCAL or client.models.list().data[0].id
print("backend:", BACKEND, "| model:", active_model)

In [ ]:
def tag_slide(idx):
    b64 = base64.standard_b64encode(png[idx].read_bytes()).decode()
    grounding = (texts.get(idx, "").strip() or "(no text elements -- image-only slide)")[:6000]
    user_text = f"Exact text from this slide:\n{grounding}\n\n{PROMPT}"

    if BACKEND == "anthropic":
        resp = client.messages.create(
            model=active_model, max_tokens=MAX_TOKENS, temperature=TEMPERATURE,
            messages=[{"role": "user", "content": [
                {"type": "image", "source": {"type": "base64",
                 "media_type": "image/png", "data": b64}},
                {"type": "text", "text": user_text}]}])
        raw = "".join(b.text for b in resp.content if b.type == "text")
    else:
        kw = dict(model=active_model, temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
                  messages=[{"role": "user", "content": [
                      {"type": "text", "text": user_text},
                      {"type": "image_url", "image_url":
                          {"url": f"data:image/png;base64,{b64}"}}]}])
        if USE_STRUCTURED_OUTPUT:
            kw["response_format"] = {"type": "json_schema", "json_schema":
                {"name": "slide", "strict": False, "schema": SLIDE_SCHEMA}}
        resp = client.chat.completions.create(**kw)
        raw = resp.choices[0].message.content

    rec = parse_json(raw)
    rec["slide_index"] = idx
    rec["exact_text"] = grounding   # verbatim Slides text fed to the model, kept for auditing
    return rec

# --- test on one slide ---
import pprint
pprint.pp(tag_slide(TEST_SLIDE))

### 6. Run the full pass (resumable)
Re-run after any interruption — it skips slides already in the JSONL. Local runs
are slow (seconds per slide); a few hundred slides can take a while.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

jsonl = WORKDIR / "slides.jsonl"
done = set()
if jsonl.exists():
    for line in jsonl.read_text().splitlines():
        try: done.add(json.loads(line)["slide_index"])
        except Exception: pass
todo = [i for i in png if i not in done]
print(f"{len(done)} done, {len(todo)} to go")

with jsonl.open("a") as fh, ThreadPoolExecutor(CONCURRENCY) as ex:
    futs = {ex.submit(tag_slide, i): i for i in todo}
    for n, fut in enumerate(as_completed(futs), 1):
        i = futs[fut]
        try:
            fh.write(json.dumps(fut.result()) + "\n"); fh.flush()
        except Exception as e:
            print(f"slide {i} FAILED: {e}")
        if n % 25 == 0:
            print(f"{n}/{len(todo)}")
print("done")

### 7. Build the table + forward-fill session dates
Each non-marker slide inherits the most recent marker's date. Warns on any backwards date jump (a misread or out-of-order slide).

In [ ]:
import re
import pandas as pd

# === manual speaker corrections (edit these — they're the reproducible knobs) ===
# 1) Per-slide overrides, keyed by slide_index. Highest priority. Use for one-off
#    misreads the rules below can't express. None drops a bogus speaker.
SPEAKER_OVERRIDES = {
    129: None,   # "Who?" placeholder
    240: None,   # "Name" template placeholder
}
# 2) Canonical aliases, keyed by the RAW speaker string (lowercased). Applies
#    everywhere that string appears — use for multi-speaker slides and any name
#    the generic "first name only" rule would mangle.
SPEAKER_ALIASES = {
    "k’lila, angel, mel": "K’lila, Angel, Melanie",
    "k’lila/angel":       "K’lila & Angel",
    "meg (and angel)":    "Meg & Angel",
    "melanie & heather":  "Melanie & Heather",
}
# signals that a string names more than one speaker (so DON'T collapse to one token)
MULTI_SPEAKER = re.compile(r"\s*(?:&|/|,|\band\b|\+)\s*", re.I)

def canon_speaker(idx, raw):
    if idx in SPEAKER_OVERRIDES:
        return SPEAKER_OVERRIDES[idx]
    s = (raw or "").strip()
    if not s:
        return None
    if s.lower() in SPEAKER_ALIASES:
        return SPEAKER_ALIASES[s.lower()]
    if MULTI_SPEAKER.search(s):                       # multi-speaker, not aliased
        parts = [p.strip().split()[0] for p in MULTI_SPEAKER.split(s) if p.strip()]
        return " & ".join(dict.fromkeys(parts))       # first name each, dedup, keep order
    return s.split()[0]                               # single speaker -> first name only
# ================================================================================

rows = [json.loads(l) for l in jsonl.read_text().splitlines() if l.strip()]
rows.sort(key=lambda r: r["slide_index"])

DATE_RE = re.compile(r"^\d{4}-\d{2}$")

# --- reconcile model misclassifications -------------------------------------
# In this deck a genuine session marker ALWAYS carries a YYYY-MM date. Anything
# flagged is_marker WITHOUT a valid date is a per-talk title card the model
# mistook for a session divider (it tended to drop the presenter's name into
# session_label). Demote those to talk slides; rescue a stranded name into the
# speaker field; never let a talk slide carry a session_label.
def is_real_marker(r):
    return bool(r.get("is_marker") and r.get("session_date")
                and DATE_RE.match(r["session_date"]))

demoted = rescued = 0
for r in rows:
    if r.get("is_marker") and not is_real_marker(r):
        r["is_marker"] = False
        demoted += 1
        lab = (r.get("session_label") or "").strip()
        if lab and not r.get("speaker") and "\n" not in lab and len(lab) <= 25:
            r["speaker"] = lab          # likely a presenter name, not a session
            rescued += 1
        r["session_label"] = None

# The model also sometimes puts a bare presenter name in `title`. Build a roster
# of every name that appears as a speaker, then move a title that EXACTLY matches
# a known speaker (and has no speaker yet) into speaker -- this leaves real
# titles ("Data Sonification", "Baroque Guitar", a URL) untouched.
roster = {(r.get("speaker") or "").strip().lower() for r in rows if r.get("speaker")}
roster.discard("")
retitled = 0
for r in rows:
    t = (r.get("title") or "").strip()
    if t and not r.get("speaker") and t.lower() in roster:
        r["speaker"] = t
        r["title"] = None
        retitled += 1

# normalize speaker names last, after every field has settled
for r in rows:
    r["speaker"] = canon_speaker(r["slide_index"], r.get("speaker"))

# strip speaker names that leaked into topics/keywords/tech_mentioned (e.g. slide 70 "Heather")
name_set = {n.lower() for r in rows if r["speaker"]
            for n in re.split(r"\s*[&,]\s*", r["speaker"]) if n}
for r in rows:
    for fld in ("topics", "keywords", "tech_mentioned"):
        r[fld] = [x for x in (r.get(fld) or []) if x.strip().lower() not in name_set]
print(f"reconciled: {demoted} false markers demoted, "
      f"{rescued} labels + {retitled} titles -> speaker; "
      f"{len({r['speaker'] for r in rows if r['speaker']})} distinct speakers")

# --- forward-fill session date/label from REAL (dated) markers only ---------
cur_label = cur_date = prev_date = None
for r in rows:
    if is_real_marker(r):
        cur_label = r.get("session_label") or cur_label
        cur_date = r["session_date"]
        if prev_date and cur_date < prev_date:
            print(f"[check] slide {r['slide_index']}: {cur_date} < prev {prev_date}")
        prev_date = cur_date
    r["session_label_ff"] = cur_label
    r["session_date_ff"] = cur_date

df = pd.DataFrame(rows)
# real datetime for the day presented; assume the 15th of the session's month
df["present_date"] = df["session_date_ff"].apply(
    lambda d: pd.Timestamp(f"{d}-15") if isinstance(d, str) and DATE_RE.match(d)
    else pd.NaT)
for col in ("topics", "keywords", "tech_mentioned"):       # list -> ";"-joined for CSV
    if col in df.columns:
        df[col] = df[col].apply(lambda x: ";".join(x or []) if isinstance(x, list) else (x or ""))
df.to_csv(WORKDIR / "slides.csv", index=False)
print(f"wrote {len(df)} rows -> {WORKDIR / 'slides.csv'} "
      f"({sum(is_real_marker(r) for r in rows)} markers, "
      f"{sum(not r['is_marker'] for r in rows)} talks, "
      f"{df['present_date'].isna().sum()} rows w/o date)")
df.head()

### 7b. (optional) Reload manual edits from slides.csv
`slides.csv` is normally an *output* of step 7. If you hand-edited it (fixed `is_marker` flags, filled in speakers, …), run **this cell instead of step 7** to pull those edits into `df` for the views below. ⚠️ Don't re-run step 7 afterward — it rebuilds from `slides.jsonl` and would clobber your edits (back up the CSV first if unsure).

In [68]:
import pandas as pd
df = pd.read_csv(WORKDIR / "slides.csv")
df["is_marker"]    = df["is_marker"].astype(str).str.strip().str.lower().isin({"true", "1"})
df["present_date"] = pd.to_datetime(df["present_date"], errors="coerce")
for col in ("topics", "keywords", "tech_mentioned", "summary"):   # empty cells -> "" not NaN
    if col in df.columns:
        df[col] = df[col].fillna("")
print(f"loaded {len(df)} rows from {WORKDIR / 'slides.csv'} "
      f"({int(df['is_marker'].sum())} markers, {int((~df['is_marker']).sum())} talks, "
      f"{int(df['speaker'].notna().sum())} with a speaker)")
df.head()

loaded 240 rows from lt_work/slides.csv (46 markers, 194 talks, 195 with a speaker)


,is_marker,session_label,session_date,speaker,title,summary,content_type,topics,keywords,tech_mentioned,legible,slide_index,exact_text,session_label_ff,session_date_ff,present_date
0,True,NaN,NaN,NaN,Research & Analytics Lightning talks,This slide announces a session on Research & A...,screenshot,data & databases;policy & government,research;analytics;lightning talks,,True,1,Research & Analytics \nLightning talks,NaN,NaN,NaT
1,True,NaN,2022-05,NaN,NaN,"This slide announces the session for May 2022,...",text,other,,,True,2,May 2022,NaN,2022-05,2022-05-15
2,False,NaN,NaN,Sara,NaN,The slide discusses Sara's experience attendin...,mixed,data visualization;urban planning & transit,workshop;urban design;data visualization;startup,,True,3,Sara\nAttended a workshop and visited the offi...,NaN,2022-05,2022-05-15
3,False,NaN,NaN,Heather,NaN,Heather Bree discusses her experience at a con...,text,data & databases;tools & workflows,metadata;analytics project;Datasheets for Data...,,True,4,Heather Bree\nI went to a conference and it re...,NaN,2022-05,2022-05-15
4,False,NaN,NaN,Angel,NaN,The slide discusses Angel Aliseda's experience...,text,data visualization;tools & workflows,Power BI;visualization;Excel;R;Python,Power BI;myLearning;Plotly;Asana;R;Python;Powe...,True,5,Angel Aliseda\nI attended the myLearning Power...,NaN,2022-05,2022-05-15


### 8. Quick exploratory peeks
First-pass signal. `topics` are now a controlled vocabulary (clean to chart directly);
the free-form `keywords` hold the specifics. Watch the **`other`** topic count — if
it's high, the `TOPICS` list in step 5 is missing a bucket.

In [ ]:
talks_only = df[~df["is_marker"]]

print("Content types:")
print(df["content_type"].value_counts(), "\n")

print("Topics (controlled vocab, talks only):")
topics = talks_only["topics"].str.split(";").explode()
print(topics[topics != ""].value_counts(), "\n")

print("Top keywords (free-form specifics):")
kw = talks_only["keywords"].str.split(";").explode()
print(kw[kw != ""].value_counts().head(20), "\n")

print("Most-mentioned tech:")
tech = df["tech_mentioned"].str.split(";").explode()
print(tech[tech != ""].value_counts().head(15))

### 9. Build the talk graph
Drop the cover slide and month markers, then wire each remaining **talk** to its
**speaker(s)**, **topics**, and **tech**. Node ids are type-prefixed (`talk:`,
`speaker:`, `topic:`, `tech:`) so a topic and a tool sharing a name stay distinct;
topic/tech keys are case-folded so `Python`/`python` collapse to one node. Exports
GraphML for Gephi / Cytoscape.

In [ ]:
import networkx as nx
from collections import Counter

# talks only: drop the cover slide (index 1) and the month-marker slides
talks = df[(df["slide_index"] != 1) & (~df["is_marker"])]

G = nx.Graph()
split_speakers = lambda s: [p.strip() for p in re.split(r"\s*[&,]\s*", str(s)) if p.strip()]

def link(talk_id, value, ntype, fold=False):
    """Add a typed node (creating it once) and connect it to the talk."""
    nid = f"{ntype}:{value.lower() if fold else value}"
    if nid not in G:
        G.add_node(nid, type=ntype, label=value)
    G.add_edge(nid, talk_id)

for _, r in talks.iterrows():
    tid = f"talk:{r['slide_index']}"
    G.add_node(tid, type="talk",
               label=(r["title"] if pd.notna(r["title"]) else "(untitled)"),
               slide_index=int(r["slide_index"]),
               date=(r["present_date"].date().isoformat()
                     if pd.notna(r["present_date"]) else ""))
    if pd.notna(r["speaker"]):                       # 26 talks have no speaker -> no edge
        for sp in split_speakers(r["speaker"]):
            link(tid, sp, "speaker")
    for tp in str(r["topics"]).split(";"):
        if tp.strip():
            link(tid, tp.strip(), "topic", fold=True)
    for tc in str(r["tech_mentioned"]).split(";"):
        if tc.strip():
            link(tid, tc.strip(), "tech", fold=True)

counts = Counter(d["type"] for _, d in G.nodes(data=True))
print("nodes:", dict(counts), "| edges:", G.number_of_edges())
for t in ("speaker", "topic", "tech"):
    top = sorted(((G.degree(n), G.nodes[n]["label"])
                  for n, d in G.nodes(data=True) if d["type"] == t), reverse=True)[:8]
    print(f"\ntop {t}s by # talks:")
    for deg, lab in top:
        print(f"  {deg:3}  {lab}")

nx.write_graphml(G, WORKDIR / "lightning_talks.graphml")
print(f"\nwrote graph -> {WORKDIR / 'lightning_talks.graphml'}")

### 10. Interactive view (pyvis)
Render `G` as a pan/zoom/drag graph. Nodes are colored by type and sized by how
many talks they touch; hover for details. Writes a self-contained HTML (vis.js
inlined, so it works offline) and embeds it inline. Drag a node to pin a cluster;
use the controls to freeze physics once it settles.

In [ ]:
from pyvis.network import Network
from IPython.display import IFrame

COLORS = {"talk": "#9aa0a6", "speaker": "#e8554e", "topic": "#4e79e8", "tech": "#3fae6b"}

net = Network(height="750px", width="100%", bgcolor="#ffffff", font_color="#222",
              notebook=True, cdn_resources="in_line")
net.barnes_hut(gravity=-9000, central_gravity=0.3, spring_length=120)
net.show_buttons(filter_=["physics"])   # tweak/freeze layout from the UI

for n, d in G.nodes(data=True):
    t, deg = d["type"], G.degree(n)
    size = 10 if t == "talk" else 8 + 3 * deg
    if t == "talk":
        tip = d["label"] + (f"  ·  {d['date']}" if d.get("date") else "")
    else:
        tip = f"{d['label']}  ·  {t} — {deg} talk{'s' if deg != 1 else ''}"
    # label talks lightly (they're many); always label speaker/topic/tech
    net.add_node(n, label=("" if t == "talk" else d["label"]),
                 title=tip, color=COLORS[t], size=size)

for u, v in G.edges():
    net.add_edge(u, v, color="#dcdcdc")

out = WORKDIR / "lightning_talks.html"
net.save_graph(str(out))
print(f"wrote {out}  —  legend: speaker=red, topic=blue, tech=green, talk=grey")
IFrame(src=str(out), width="100%", height="780px")

### 11. Speaker ↔ topic relationships
Collapse the talk in the middle: link each **speaker** straight to the **topics**
they presented, with edge weight = how many of their talks carried that topic (so
`Heather —[8]— data visualization` means 8 of her talks were tagged data viz). The
full weighted list is written to `speaker_topic.csv`; the graph keeps only topics
seen in ≥ `TOPIC_MIN_TALKS` talks (the 400+ one-off tags are noise — see step 8).
Edge thickness = number of shared talks.

In [ ]:
import networkx as nx
from collections import Counter
from itertools import product
from pyvis.network import Network
from IPython.display import IFrame, display

TOPIC_MIN_TALKS = 2   # drop one-off topics from the graph; set 1 to keep all 400+
split_speakers = lambda s: [p.strip() for p in re.split(r"\s*[&,]\s*", str(s)) if p.strip()]

td = df[(df["slide_index"] != 1) & (~df["is_marker"])]

# weighted speaker–topic co-occurrence (one count per talk linking each speaker to each topic)
st, topic_total, speaker_total = Counter(), Counter(), Counter()
for _, r in td.iterrows():
    if pd.isna(r["speaker"]):
        continue
    sps = split_speakers(r["speaker"])
    tps = {t.strip().lower() for t in str(r["topics"]).split(";") if t.strip()}
    for sp in sps:
        speaker_total[sp] += 1
    for tp in tps:
        topic_total[tp] += 1
    for sp, tp in product(sps, tps):
        st[(sp, tp)] += 1

# full weighted list -> CSV + on-screen
edges_df = (pd.DataFrame([(sp, tp, w) for (sp, tp), w in st.items()],
                         columns=["speaker", "topic", "talks"])
            .sort_values("talks", ascending=False, ignore_index=True))
edges_df.to_csv(WORKDIR / "speaker_topic.csv", index=False)
print(f"{len(edges_df)} speaker–topic links -> {WORKDIR / 'speaker_topic.csv'}")
display(edges_df.head(15))

# bipartite graph, recurring topics only
keep = {tp for tp, n in topic_total.items() if n >= TOPIC_MIN_TALKS}
B = nx.Graph()
for sp in speaker_total:
    B.add_node(f"speaker:{sp}", type="speaker", label=sp)
for tp in keep:
    B.add_node(f"topic:{tp}", type="topic", label=tp)
for (sp, tp), w in st.items():
    if tp in keep:
        B.add_edge(f"speaker:{sp}", f"topic:{tp}", weight=w)
print(f"graph: {len(speaker_total)} speakers, {len(keep)} topics "
      f"(>= {TOPIC_MIN_TALKS} talks), {B.number_of_edges()} links")

net = Network(height="750px", width="100%", bgcolor="#ffffff", font_color="#222",
              notebook=True, cdn_resources="in_line")
net.barnes_hut(gravity=-12000, central_gravity=0.25, spring_length=150)
net.show_buttons(filter_=["physics"])
for n, d in B.nodes(data=True):
    is_sp = d["type"] == "speaker"
    tot = speaker_total[d["label"]] if is_sp else topic_total[d["label"]]
    net.add_node(n, label=d["label"], title=f"{d['label']} — {tot} talks",
                 color="#e8554e" if is_sp else "#4e79e8",
                 size=(12 + 2 * tot) if is_sp else (10 + 3 * tot))
for u, v, d in B.edges(data=True):
    net.add_edge(u, v, value=d["weight"], title=f"{d['weight']} talks", color="#c8c8c8")

out = WORKDIR / "speaker_topic.html"
net.save_graph(str(out))
print(f"wrote {out}  —  speaker=red, topic=blue, edge width = shared talks")
IFrame(src=str(out), width="100%", height="780px")

### 12. Speaker activity timeline
One row per speaker; the line runs from their **first to last** talk, each dot is a
talk, and the number is their total. This shows tenure-in-context honestly — a 4-year
regular vs. a one-session guest — without faking a per-speaker *rate*. We deliberately
**don't** normalize by rate: observed span is only a lower bound on real tenure (we
see talks, not employment dates), and dividing a tiny count by a short span makes
one-talk speakers look hyper-active. To get a true rate you'd need real start/end
dates (an HR input we don't have).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

split_speakers = lambda s: [p.strip() for p in re.split(r"\s*[&,]\s*", str(s)) if p.strip()]

tl = df[(df["slide_index"] != 1) & (~df["is_marker"])
        & df["speaker"].notna() & df["present_date"].notna()]

# explode multi-speaker talks; collect each speaker's talk dates
events = {}
for _, r in tl.iterrows():
    for sp in split_speakers(r["speaker"]):
        events.setdefault(sp, []).append(r["present_date"])

# rows ordered by first appearance (earliest at top)
speakers = sorted(events, key=lambda s: min(events[s]))

fig, ax = plt.subplots(figsize=(11, 0.42 * len(speakers) + 1))
for y, sp in enumerate(speakers):
    ds = sorted(events[sp])
    ax.plot([ds[0], ds[-1]], [y, y], "-", color="#c8c8c8", lw=2, zorder=1)
    ax.scatter(ds, [y] * len(ds), s=28, color="#4e79e8", zorder=2)
    ax.text(ds[-1], y, f"  {len(ds)}", va="center", fontsize=8, color="#444")

ax.set_yticks(range(len(speakers)))
ax.set_yticklabels(speakers, fontsize=9)
ax.invert_yaxis()                                    # earliest speaker on top
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.grid(axis="x", color="#eee")
ax.margins(x=0.03)
ax.set_title("Speaker activity — first→last talk (dot = a talk, number = total)")
plt.tight_layout()
plt.savefig(WORKDIR / "speaker_timeline.png", dpi=130)
plt.show()
print(f"{len(speakers)} speakers -> {WORKDIR / 'speaker_timeline.png'}")

### 13. Most wide-ranging speakers
Top speakers by **variety** = how many distinct controlled `topics` they've covered.
Talk count is shown alongside on purpose: with only ~17 buckets, variety partly just
tracks volume (more talks → more chances to span topics), so read the two together.

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

TOP_N = 5
split_speakers = lambda s: [p.strip() for p in re.split(r"\s*[&,]\s*", str(s)) if p.strip()]

tv = df[(df["slide_index"] != 1) & (~df["is_marker"]) & df["speaker"].notna()]

topics_by_speaker, talks_by_speaker = {}, Counter()
for _, r in tv.iterrows():
    tops = {t.strip() for t in str(r["topics"]).split(";") if t.strip() and t.strip() != "other"}
    for sp in split_speakers(r["speaker"]):
        topics_by_speaker.setdefault(sp, set()).update(tops)
        talks_by_speaker[sp] += 1

# rank by distinct topics, then by talks as tiebreak
rank = sorted(topics_by_speaker.items(),
              key=lambda kv: (-len(kv[1]), -talks_by_speaker[kv[0]]))

print(f"Top {TOP_N} speakers by topic variety:\n")
for sp, tops in rank[:TOP_N]:
    print(f"  {sp:12} {len(tops):2} topics / {talks_by_speaker[sp]:2} talks: {', '.join(sorted(tops))}")

top = rank[:TOP_N][::-1]
fig, ax = plt.subplots(figsize=(8, 0.6 * TOP_N + 1))
ax.barh([s for s, _ in top], [len(t) for _, t in top], color="#3fae6b")
for y, (sp, tops) in enumerate(top):
    ax.text(len(tops), y, f"  {len(tops)} ({talks_by_speaker[sp]} talks)",
            va="center", fontsize=9, color="#444")
ax.set_xlabel("distinct topics covered")
ax.set_title(f"Top {TOP_N} most wide-ranging speakers")
ax.margins(x=0.12)
plt.tight_layout()
plt.savefig(WORKDIR / "speaker_variety.png", dpi=130)
plt.show()

### 14. Breadth independent of volume (Simpson diversity)
For each speaker with ≥ `MIN_TALKS` talks, **Simpson's diversity** = the probability
that two of their talks (drawn at random) land on *different* controlled topics. It
captures how *evenly* someone spreads across themes rather than how many talks they
gave: **1.0 ≈ ranges widely, 0.0 = one-note specialist**. The threshold drops speakers
with too few talks to estimate. Sorted high→low, so it doubles as a generalist↔specialist
spectrum.

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

MIN_TALKS = 5   # below this, too few talks to estimate diversity reliably
split_speakers = lambda s: [p.strip() for p in re.split(r"\s*[&,]\s*", str(s)) if p.strip()]

tv = df[(df["slide_index"] != 1) & (~df["is_marker"]) & df["speaker"].notna()]

# topic-occurrence counts per speaker (a talk contributes each of its topics)
topic_counts, talks_by = {}, Counter()
for _, r in tv.iterrows():
    tops = [t.strip() for t in str(r["topics"]).split(";") if t.strip() and t.strip() != "other"]
    for sp in split_speakers(r["speaker"]):
        talks_by[sp] += 1
        topic_counts.setdefault(sp, Counter()).update(tops)

def simpson(counter):
    """Gini–Simpson: P(two random topic-tags differ). 0 = specialist, ->1 = broad."""
    tot = sum(counter.values())
    return 0.0 if tot <= 1 else 1 - sum((n / tot) ** 2 for n in counter.values())

ranked = sorted(
    ((sp, talks_by[sp], len(c), simpson(c)) for sp, c in topic_counts.items()
     if talks_by[sp] >= MIN_TALKS),
    key=lambda x: -x[3])

print(f"Breadth (Simpson diversity), speakers with >= {MIN_TALKS} talks:\n")
print(f"  {'speaker':12} {'talks':>5} {'topics':>7} {'diversity':>10}")
for sp, n, d, s in ranked:
    print(f"  {sp:12} {n:5} {d:7} {s:10.3f}")

fig, ax = plt.subplots(figsize=(8, 0.5 * len(ranked) + 1))
labels = [sp for sp, *_ in ranked][::-1]
vals = [s for *_, s in ranked][::-1]
talks = [n for _, n, _, _ in ranked][::-1]
bars = ax.barh(labels, vals, color=plt.cm.viridis([v for v in vals]))
for y, (v, n) in enumerate(zip(vals, talks)):
    ax.text(v, y, f"  {v:.2f} ({n} talks)", va="center", fontsize=8, color="#333")
ax.set_xlim(0, 1)
ax.set_xlabel("Simpson diversity  (0 = specialist · 1 = wide-ranging)")
ax.set_title(f"Topic breadth independent of volume (>= {MIN_TALKS} talks)")
ax.margins(x=0.15)
plt.tight_layout()
plt.savefig(WORKDIR / "speaker_breadth.png", dpi=130)
plt.show()

### 14b. Speaker blurbs (local LLM)
For each speaker, feed their own talk titles + summaries to the local model and ask for a cute, SFW, gently-funny one- or two-sentence description. Cached to `speaker_blurbs.json` (edit/delete to retune; only speakers missing a blurb are generated unless `REGEN_ALL`). Shown on the speaker's hover popup in step 15 — **re-run step 15 after this** to bake them in.

In [ ]:
import json, re
BLURBS_PATH = WORKDIR / "speaker_blurbs.json"
BLURB_TEMP = 0.8
REGEN_ALL = False   # True = regenerate everyone; False = only speakers missing a blurb

split_speakers = lambda s: [p.strip() for p in re.split(r"\s*[&,]\s*", str(s)) if p.strip()]

# each speaker's own talk titles + (trimmed) summaries -> the LLM's material
talks_only = df[(df["slide_index"] != 1) & (~df["is_marker"]) & df["speaker"].notna()]
corpus = {}
for _, r in talks_only.iterrows():
    title = (r["title"] if pd.notna(r.get("title")) else "").strip()
    summ  = (r["summary"] if pd.notna(r.get("summary")) else "").strip()[:140]
    line  = f"- {title + ': ' if title else ''}{summ}".strip()
    for sp in split_speakers(r["speaker"]):
        corpus.setdefault(sp, []).append(line)

def make_blurb(name, lines):
    talks_text = "\n".join(lines[:20]) or "(no talk details)"
    msg = (f"You're writing a playful one-liner about a colleague from the lightning talks "
           f"they've given. Keep it warm, gently funny, and fully workplace-appropriate (SFW) — "
           f"affectionate, never mean.\n\n{name}'s talks:\n{talks_text}\n\n"
           f"Write 1–2 sentences describing {name} as a presenter, riffing on the range and themes "
           f"of their talks. Output ONLY the description — no preamble, no quotes.")
    if BACKEND == "anthropic":
        resp = client.messages.create(model=active_model, max_tokens=120, temperature=BLURB_TEMP,
                                      messages=[{"role": "user", "content": msg}])
        out = "".join(b.text for b in resp.content if b.type == "text")
    else:
        resp = client.chat.completions.create(model=active_model, max_tokens=160, temperature=BLURB_TEMP,
                                              messages=[{"role": "user", "content": msg}])
        out = resp.choices[0].message.content
    return re.sub(r"<think>.*?</think>", "", out, flags=re.S).strip().strip('"').strip()

blurbs = {} if REGEN_ALL else (json.loads(BLURBS_PATH.read_text()) if BLURBS_PATH.exists() else {})
todo = [sp for sp in corpus if sp not in blurbs]
print(f"{len(blurbs)} cached, generating {len(todo)} with {active_model} ...")
for sp in todo:
    try:
        blurbs[sp] = make_blurb(sp, corpus[sp])
        BLURBS_PATH.write_text(json.dumps(blurbs, indent=2, ensure_ascii=False))   # save as we go
        print(f"  • {sp}: {blurbs[sp]}")
    except Exception as e:
        print(f"  • {sp} FAILED: {e}")
print(f"\nwrote {len(blurbs)} blurbs -> {BLURBS_PATH} (edit/delete to taste; then re-run step 15)")


### 14c. Rework talk descriptions
Revise each shown talk's `summary` so it describes the talk directly instead of the slide (“This slide announces…” → plain description). Cached to `summary_revised.json` (editable; only un-revised talks are processed unless `REGEN_ALL_SUMMARIES`). The step-15 data cell prefers the revised text. The step-5 prompt is also fixed so future tagging is clean.

In [ ]:
import json, re
REV_PATH = WORKDIR / "summary_revised.json"
REV_TEMP = 0.3
REGEN_ALL_SUMMARIES = False   # True = redo all; False = only summaries not yet revised

def _has_topic(s):
    return any(t.strip() and t.strip() != "other" for t in str(s).split(";"))

# the talks shown in the 3D view (non-marker, with speaker + date + a real topic)
shown = df[(df["slide_index"] != 1) & (~df["is_marker"])
           & df["speaker"].notna() & df["present_date"].notna()]
shown = shown[shown["topics"].map(_has_topic)]

def revise(summary, title):
    head = f"Title: {title}\n" if title else ""
    msg = ("Rewrite this lightning-talk description so it describes the talk's content directly. "
           "Strip any meta-language about the slide itself — never say 'this slide', 'this presentation', "
           "'the slide shows/announces', 'serves as a title card', etc. Keep the facts, present tense, "
           "1–2 sentences. Output ONLY the rewritten description — no preamble, no quotes.\n\n"
           f"{head}Description: {summary}")
    if BACKEND == "anthropic":
        r = client.messages.create(model=active_model, max_tokens=160, temperature=REV_TEMP,
                                   messages=[{"role": "user", "content": msg}])
        out = "".join(b.text for b in r.content if b.type == "text")
    else:
        r = client.chat.completions.create(model=active_model, max_tokens=200, temperature=REV_TEMP,
                                           messages=[{"role": "user", "content": msg}])
        out = r.choices[0].message.content
    return re.sub(r"<think>.*?</think>", "", out, flags=re.S).strip().strip('"').strip()

rev = {} if REGEN_ALL_SUMMARIES else (json.loads(REV_PATH.read_text()) if REV_PATH.exists() else {})
todo = [r for _, r in shown.iterrows()
        if str(int(r["slide_index"])) not in rev and str(r["summary"]).strip()]
print(f"{len(rev)} cached, revising {len(todo)} summaries with {active_model} ...")
for r in todo:
    sid = str(int(r["slide_index"]))
    title = r["title"] if pd.notna(r.get("title")) else ""
    try:
        rev[sid] = revise(str(r["summary"]), title)
        REV_PATH.write_text(json.dumps(rev, indent=2, ensure_ascii=False))   # save as we go
        if len(rev) % 20 == 0:
            print(f"  ...{len(rev)} done")
    except Exception as e:
        print(f"  slide {sid} FAILED: {e}")
print(f"wrote {len(rev)} revised summaries -> {REV_PATH} (edit/delete to taste; then re-run step 15)")


### 15. Talk space (3D)
Talks in a 3D space — **x = date · y = speaker (each a slice) · z = primary topic** (least→most technical, `other` excluded) — rendered in **Three.js**: smooth `OrbitControls` orbit, raycaster hit-testing for a reliable click, and CSS2D billboard labels. People are gold **stars**. Hover shows the talk description; **click** a talk or a star to lock the highlight (click again / another to toggle); click a **z-axis topic label** to highlight all talks on that theme. Dark background. Loads Three.js from a CDN (needs internet); writes a self-contained page — open `lt_work/talk_space_three.html` directly if the inline iframe is blocked.

In [69]:
import json
from collections import Counter
from IPython.display import IFrame

# This cell only emits the DATA (lt_work/talk_space_data.js). The view itself lives in
# the editable static page ./talk_space_three.html, which loads that data file.
split_speakers = lambda s: [p.strip() for p in re.split(r"\s*[&,]\s*", str(s)) if p.strip()]
def primary_topic(s):
    for t in str(s).split(";"):
        t = t.strip()
        if t and t != "other":
            return t
    return None

tg = df[(df["slide_index"] != 1) & (~df["is_marker"])
        & df["speaker"].notna() & df["present_date"].notna()]
_rp = WORKDIR / "summary_revised.json"
_rev = json.loads(_rp.read_text()) if _rp.exists() else {}
recs = []
for _, r in tg.iterrows():
    pt = primary_topic(r["topics"])
    if not pt:
        continue
    d = r["present_date"]
    recs.append(dict(idx=int(r["slide_index"]), speakers=split_speakers(r["speaker"]), pt=pt, x=d.year + (d.month - 1) / 12.0,
                     date=d.date().isoformat(),
                     summary=_rev.get(str(int(r["slide_index"])), (r["summary"] if pd.notna(r.get("summary")) else ""))))

present = {t["pt"] for t in recs}
topics_sorted = [t for t in TOPICS if t != "other" and t in present]
topics_sorted += sorted(present - set(topics_sorted))
level = {t: i for i, t in enumerate(topics_sorted)}
first_x, sp_topics, sp_xs = {}, {}, {}
for t in recs:
    for sp in t["speakers"]:
        first_x[sp] = min(first_x.get(sp, t["x"]), t["x"])
        sp_topics.setdefault(sp, Counter())[t["pt"]] += 1
        sp_xs.setdefault(sp, []).append(t["x"])
speakers_ordered = sorted(first_x, key=lambda s: first_x[s])
ypos = {sp: i for i, sp in enumerate(speakers_ordered)}
sp_z = {sp: sum(level[p] * c for p, c in cnt.items()) / sum(cnt.values()) for sp, cnt in sp_topics.items()}
sp_xm = {sp: sum(xs) / len(xs) for sp, xs in sp_xs.items()}

_bp = WORKDIR / "speaker_blurbs.json"
_blurbs = json.loads(_bp.read_text()) if _bp.exists() else {}
data = dict(
    xmin=min(t["x"] for t in recs), xmax=max(t["x"] for t in recs),
    nY=len(speakers_ordered), nZ=len(topics_sorted), topics=topics_sorted,
    talks=[dict(idx=t["idx"], x=t["x"], y=ypos[sp], z=level[t["pt"]],
                sp=sp, spk=", ".join(t["speakers"]), date=t["date"], topic=t["pt"], summary=t["summary"])
           for t in recs for sp in t["speakers"]],
    speakers=[dict(x=sp_xm[s], y=ypos[s], z=sp_z[s], name=s, talks=sum(sp_topics[s].values()), blurb=_blurbs.get(s, ""))
              for s in speakers_ordered],
    edges=[dict(speaker=sp, segs=[[sp_xm[sp], ypos[sp], sp_z[sp], t["x"], ypos[sp], level[t["pt"]]]
                                  for t in recs if sp in t["speakers"]])
           for sp in speakers_ordered])

DOCS = Path("docs"); DOCS.mkdir(exist_ok=True)
(DOCS / "talk_space_data.js").write_text("window.TALK_DATA = " + json.dumps(data) + ";")
print(f"wrote {DOCS / 'talk_space_data.js'}  "
      f"({len(data['talks'])} points, {len(data['speakers'])} speakers, {len(topics_sorted)} topics)")
print("view / publish source: ./docs/index.html  (edit it for visuals; commit docs/ to GitHub Pages)")
IFrame(src="docs/index.html", width="100%", height="840px")


wrote lt_work/talk_space_data.js  (188 points, 26 speakers, 17 topics)
view: ./talk_space_three.html  (edit that file for visuals; rerun this cell after re-tagging)
